In [ ]:
from google.colab import userdata
KEY_VALUE = 'MY_API_KEY' # userdata.get('MY_KEY')

# MODEL_ID = "Claude-Sonnet-4"
MODEL_ID = "Gemini-2.5-Flash"
API_ENDPOINT="https://api.poe.com/v1"

# Using Google API key via LiteLLM
# MODEL_ID = "gemini/gemini-2.5-flash"

POSTPEND_STRING = '' # '--thinking_budget 0 --web_search false'
GLOBAL_EXECUTOR = 'exec'
MAX_TOKENS = 64000

In [ ]:
!apt-get update
!apt-get install fpc fpc-source lazarus

In [ ]:
!pip install duckduckgo-search

In [ ]:
!git clone -b v1.23-bp https://github.com/joaopauloschuler/beyond-python-smolagents smolagents

In [ ]:
!pip install ./smolagents[openai]

In [ ]:
import smolagents
from smolagents.bp_tools import *
from smolagents.bp_utils import *
from smolagents.bp_thinkers import *
from smolagents import LogLevel
from smolagents import OpenAIServerModel
from smolagents import VisitWebpageTool, DuckDuckGoSearchTool
from smolagents.default_tools import DuckDuckGoSearchTool

In [ ]:
LocalVistWebPageTool = VisitWebpageTool()
LocalWebSearchTool = DuckDuckGoSearchTool()

In [ ]:
GLOBAL_LOG_LEVEL = LogLevel.DEBUG
STEP_CALLBACKS = [delay_execution_10]

# Via LiteLLM (Gemini, ...)
# model = LiteLLMModel(MODEL_ID, api_key=KEY_VALUE, max_tokens=MAX_TOKENS)

# Using OpenAI protocol
model = OpenAIServerModel(MODEL_ID, api_key=KEY_VALUE, max_tokens=MAX_TOKENS, api_base=API_ENDPOINT)
model.postpend_string = POSTPEND_STRING

model.verbose = False
additional_authorized_imports=['*']
tools = [ run_os_command,
  pascal_interface_to_string, source_code_to_string, string_to_source_code,
  save_string_to_file, load_string_from_file,
  LocalVistWebPageTool, LocalWebSearchTool,
  copy_file, replace_in_file, replace_in_file_from_files, get_file_size,
  get_line_from_file, is_file, print_file_lines
  ]

In [ ]:
!rm advices.notes
!rm *.pas
!mkdir bin
!rm bin/*
!rm *.txt

In [ ]:
computing_language = "free pascal (fpc)"
what_to_code = "task manager"
fileext='.pas'
start_now = True
refine = True

has_pascal_message = """ When compiling pascal code, use this example:
run_os_command('fpc solution1.pas -obin/task_manager -O1 -Mobjfpc')
Notice in the example above that there is no space after "-o" for the output file parameter.
With fpc, do not use -Fc nor -o/dev/null or similar.
Do not code any user input such as ReadLn. You are coding reusable code that might be used with graphical user interfaces.
You will replace fixed sized arrays by dynamic arrays.
All pascal reserved words will be typed in lowercase.
Do not change the current working folder.
When you are asked to compare solutions, compile each version/solution. Only select solutions that do compile.
When compiling code, generate your binaries at the bin/ folder. Do not mix source code with binary (compiled) files.
When testing, review the source code and test if it compiles. Verify for the risk of any infinite loop or memory leak.
Only try to run code after verifying for infinite loop, memory leak and compilation errors.
Feel free to search the internet with error messages if you need.
This is an example how to code and compile a pascal program:
<example>
<savetofile filename='solutionx.pas'>
program mytask;
{$mode objfpc} // Use Object Pascal mode for dynamic arrays and objects

uses
  SysUtils,
  DateUtils,
  mytask; // your unit

begin
  WriteLn('Hello!');
end.
</savetofile>
<runcode>
print("Attempting to compile solutionx.pas...")
compile_output = run_os_command('fpc solutionx.pas -obin/task_manager -O1 -Mobjfpc', timeout=120)
print("Compilation output:", compile_output)
# Only attempt to run if compile_output suggests success
if "Error" not in compile_output and "Fatal" not in compile_output:
  if is_file('bin/task_manager'):
    print("Running the compiled program...")
    print(run_os_command('bin/task_manager', timeout=120))
  else:
    print("Executable not found.")
else:
  print("Compilation failed.")
  import re
  error_lines = re.findall(r'solutionx\\.pas\\((\\d+),\\d+\\).*', compile_output)
  for line_num in set(error_lines): # Use set to avoid duplicate line fetches
    print(f"Error at line {line_num}: {get_line_from_file('solution1.pas', int(line_num))}")
</runcode>
</example>

Each time that you have an error such as "solutionx.pas(206,14) Fatal: Syntax error, "identifier" expected but "is" found",
you will call something like this: get_line_from_file('solutionx.pas',206)
REMEMBER:
* "```pascal" will not save a pascal file into disk. Use savetofile tags instead.
* DO NOT CREATE A SOLUTION WITH MULTIPLE FILES. Save the entire solution into solutionx.pas where x is either 1, 2 or 3.
* DO NOT declare variables within a begin/end block. ALWAYS declare variables in the declaration area.
* DO NOT use label/go to.
* DO NOT declare anything that starts with a digit such as:
   var 1stVariable: integer;
* DO NOT use the type `real` for real numbers as it depends on hardware. Use `double` or `single` instead.
* CREATE A TYPE for dynamic array function results.
  This declaration will fail: `function solve(anp: integer; var acostmatrix: array of tRealArray): array of tAppointmentResult;`.
  Do this instead: ```
  type
    TApptResultDynArr = array of tAppointmentResult;
  ...
  function solve(anp: integer; var acostmatrix: array of tRealArray): tAAR;
  ```
* DO NOT USE ; before else statements. Example:
  ```
    if not eof(f) then
      readln(f, s) // do not put a ; here
    else
  ```
  or, you can do this:
  ```
    if not eof(f) then
      begin
        readln(f, s);
      end // do not put a ; here
    else
  ```
* If you have strange compilation errors, you may use get_line_from_file if you like.
* Include in your uses the unit math as the unit math contains many useful constants and functions (such as MaxDouble).
* When passing arrays as parameter, consider passing as reference to avoid memory copying.
* Create a method called self_test. In this method, you will code static inputs for testing (there will be no externally entered data to test with - do not use ReadLn for testing).
* BE BOLD AND CODE AS MANY FEATURES AS YOU CAN!
* If any of your questions is not answered, assume your best guess. Do not keep asking or repeat questions. Just follow your best guess.
* The bin folder has already been created.
* Your goal is pascal coding. Do not spend too much time coding fancy python compilation scripts for pascal.
"""

task = """Using only the """+computing_language+""" computing language, code a """+what_to_code+" ."
task += has_pascal_message + """
Feel free to search the internet with error messages if you need.
As you are super-intelligent, I do trust you.
YOU ARE THE BRAIN OF AN AGENT INSIDE OF THE FANTASTIC BEYOND PYTHON SMOLAGENTS: https://github.com/joaopauloschuler/beyond-python-smolagents . Enjoy!
As you are the brain of an agent, this is why you are required to respond with "final_answer" at each conclusive reply from you.
"""

In [ ]:
print(task)

In [ ]:
evolutive_problem_solver(model, task, agent_steps=10, steps=24, start_now=start_now,
    fileext=fileext, tools=tools, log_level=LogLevel.ERROR, step_callbacks=[delay_execution_10], refine=refine)